In [7]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] ="2"

import tensorflow as tf
tf.config.list_physical_devices("GPU")


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [8]:


tf.config.list_physical_devices("GPU")
import warnings
warnings.filterwarnings('ignore')
from contextlib import redirect_stdout, redirect_stderr
from tensorflow import keras
import sys
import contextlib
tf.__version__

'2.10.0'

In [9]:
import gym
import numpy as np
import matplotlib.pyplot as plt

import gym
print(gym.__version__)
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from IPython import display

0.26.2


In [10]:
import tensorflow_probability as tfp


In [13]:
!nvidia-smi

Tue Oct 24 06:58:54 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:05:00.0 Off |                  N/A |
|  0%   54C    P2    45W / 198W |   7696MiB /  8192MiB |      2%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  On   | 00000000:06:00.0 Off |                  N/A |
| 27%   

In [14]:
def preprocess(image):
    """Preprocess a frame from the game."""
    image = image[35:195]  # Crop
    image = image[::2, ::2, 0]  # Downsample by a factor of 2
    image[image == 144] = 0  # Erase background (background type 1)
    image[image == 109] = 0  # Erase background (background type 2)
    image[image != 0] = 1  # Set paddles and ball to 1, everything else to 0
    return np.reshape(image.astype(np.float32).ravel(), [80, 80])


# Neural Network 

In [15]:

# hyperparameters
learning_rate = 0.0003 #0.0001
gamma = 0.99

env = gym.make("Pong-v0")

A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [16]:
env.reset()[0].shape

(210, 160, 3)

In [29]:

model_policy = tf.keras.Sequential()
model_policy.add(tf.keras.Input(shape=(80, 80, 2)))
#model_policy.add(tf.keras.layers.Conv2D(64, (8, 8), strides=(4, 4), activation='relu'))
#model_policy.add(tf.keras.layers.Conv2D(32, (4, 4), strides=(2, 2), activation='relu'))
model_policy.add(tf.keras.layers.Flatten())
model_policy.add(tf.keras.layers.Dense(256, activation='linear'))
#model_policy.add(tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.0002)))
model_policy.add(tf.keras.layers.Dense(64, activation='relu'))
model_policy.add(tf.keras.layers.Dropout(0.3))
model_policy.add(tf.keras.layers.Dense(2, activation='softmax'))
optimizer_policy = tf.keras.optimizers.Adam(learning_rate)


In [30]:

model_value = tf.keras.Sequential()
model_value.add(tf.keras.Input(shape=(80, 80, 2)))
model_value.add(tf.keras.layers.Flatten())
model_value.add(tf.keras.layers.Dense(32, activation='relu'))
model_value.add(tf.keras.layers.Dense(1, activation='linear'))

optimizer_value = tf.keras.optimizers.Adam(learning_rate)


## Load model weights from episode 1500 previously. 

In [31]:
#from tensorflow.keras.models import load_model

#model_policy = load_model('policy_model_1500')
#model_value = load_model('policy_model_' + str(batch))


## continuing the model

In [ ]:

# training loop

episode_rewards = []
for batch in range (0, 5000):

    loss_policy = 0
    mse_loss = 0
    with tf.GradientTape(persistent=True) as tape_policy:
        with tf.GradientTape(persistent=True) as tape_value:
            reward_history_eps = []
            index_history_eps = []
            prob_output_eps = tf.Variable(tf.zeros([0, 2], dtype=tf.float32))
            Vt_eps = tf.Variable(tf.zeros([0, 1], dtype=tf.float32))
            state = preprocess(env.reset()[0])
            prev_state = state
            episode_reward = 0
            terminated = False
            truncated = False 
            while (not terminated) and (not truncated):
                with open('output.log', 'w') as log_file:
                    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                        if episode_reward==0:
                            combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(prev_state)], axis=-1)
                            action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                        else: 
                            action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                        #action_prob = model_policy(np.expand_dims(prev_state, axis=0))[0]

                dist = tfp.distributions.Categorical(probs=action_prob, dtype=tf.float32)
                action = dist.sample()
                #print(int(action))
                if int(action)==0:
                    a=2
                elif int(action)==1:
                    a=3
                Vt = model_value(np.expand_dims(combined_states, axis=0))
                #print(Vt)
                Vt_eps = tf.concat([Vt_eps, Vt], axis=0)
                index_history_eps.append(int(action))
                prob_output_eps = tf.concat([prob_output_eps, tf.expand_dims(action_prob, 0)], axis=0)
                next_state, reward, terminated, truncated, info = env.step(a)
                reward_history_eps.append(reward)

                next_state = preprocess(next_state)
                episode_reward += reward
                combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(next_state)], axis=-1)

                prev_state = next_state
            #end of episode
            episode_rewards.append(episode_reward)
            
            if batch<100:
                print(f"Episode: {batch}, Reward: {episode_reward}")
            
            if len(episode_rewards) >= 100 and batch%100==0 :
                
                print(f"Episode: {batch}, Reward: {episode_reward}")
                avg_reward = np.mean(episode_rewards[-100:])
                print(f"Average Reward (Last 100 Episodes): {avg_reward}")
                    #break
    #after the end of the episode   
            discounted_rewards_eps = []
            reward_sum = 0
            for r in reward_history_eps[::-1]:
                reward_sum = r + gamma * reward_sum
                discounted_rewards_eps.insert(0, reward_sum)

            discounted_rewards_eps = np.array(discounted_rewards_eps)
            reshaped_discounted_rewards = discounted_rewards_eps.reshape(-1, 1)


            #index_history_eps = tf.cast(index_history_eps, dtype=tf.int32)  # Or tf.int64, depending on your needs

            index_history_eps =tf.transpose(tf.expand_dims(tf.cast(index_history_eps,  dtype=tf.int32), axis=0))
            tensor_rewards = tf.convert_to_tensor(reshaped_discounted_rewards, dtype=tf.float32)
            ###normalize the discounted rewards
            mean = tf.reduce_mean(tensor_rewards)
            std = tf.math.reduce_std(tensor_rewards)

            # Normalize the discounted rewards
            normalized_rewards = (tensor_rewards - mean) / (std + 1e-8)
            tensor_rewards_normalized = tf.convert_to_tensor(normalized_rewards, dtype=tf.float32)

            #loss =loss +  tf.reduce_sum (index_history_eps * tensor_rewards_normalized)
            
            for idx in range (0, tensor_rewards_normalized.shape[0]):
                loss_policy = loss_policy +  (tensor_rewards_normalized[idx]-Vt_eps[idx])* tf.math.log(prob_output_eps[idx][int(index_history_eps[idx])])
            loss_policy_mean = -tf.reduce_mean(tf.convert_to_tensor(loss_policy))
            
            #

            # Calculate the element-wise loss
            '''
            elementwise_loss = (tensor_rewards - Vt_eps) * tf.math.log(tf.gather_nd(prob_output_eps, index_history_eps, batch_dims=1))

            loss_policy_mean = -tf.reduce_mean(elementwise_loss)
            '''

            ##After one batch 
            #print(Vt_eps)
            mse_loss = tf.keras.losses.mean_squared_error(Vt_eps, tensor_rewards_normalized)

    grads_policy = tape_policy.gradient(loss_policy_mean, model_policy.trainable_variables)
    optimizer_policy.apply_gradients(zip(grads_policy, model_policy.trainable_variables))

    # Calculate the simple moving average of the last 100 episode rewards
    grads_value = tape_value.gradient(mse_loss, model_value.trainable_variables)
    optimizer_value.apply_gradients(zip(grads_value, model_value.trainable_variables))
    
    
    if batch%100==0 and batch!=0:
        model_policy.save('policy_model_'+str(batch))  # This will create a 'my_model' directory with model parameters
        model_value.save('value_model_afterloadingmodel_'+str(batch))  # This will create a 'my_model' directory with model parameters
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath="model_checkpoint.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1, )
        
        
        warnings.filterwarnings('ignore')

        episode_numbers = list(range(1, len(episode_rewards) + 1))
        moving_avg = np.convolve(episode_rewards, np.ones(100) / 100, mode='valid')
        plt.plot(episode_numbers, episode_rewards, label="Episode Reward")
        plt.plot(episode_numbers[-len(moving_avg):], moving_avg, label="Moving Average (100 episodes)")
        plt.xlabel("Episode")
        plt.ylabel("Reward")
        #plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        #plt.show()
        plt.savefig('episode_rewards_'+ str(batch)+'.png')
        np_eps_rewards = np.array(episode_rewards)
        np.save('np_eps_rewards_'+str(batch)+'.npy', np_eps_rewards)


env.close()



Episode: 0, Reward: -21.0
Episode: 1, Reward: -19.0
Episode: 2, Reward: -21.0
Episode: 3, Reward: -21.0
Episode: 4, Reward: -19.0
Episode: 5, Reward: -20.0
Episode: 6, Reward: -20.0
Episode: 7, Reward: -21.0
Episode: 8, Reward: -21.0
Episode: 9, Reward: -21.0
Episode: 10, Reward: -21.0
Episode: 11, Reward: -21.0
Episode: 12, Reward: -21.0
Episode: 13, Reward: -21.0
Episode: 14, Reward: -21.0
Episode: 15, Reward: -21.0
Episode: 16, Reward: -21.0
Episode: 17, Reward: -21.0
Episode: 18, Reward: -21.0
Episode: 19, Reward: -20.0
Episode: 20, Reward: -21.0
Episode: 21, Reward: -21.0
Episode: 22, Reward: -21.0
Episode: 23, Reward: -21.0
Episode: 24, Reward: -21.0
Episode: 25, Reward: -21.0
Episode: 26, Reward: -20.0
Episode: 27, Reward: -21.0
Episode: 28, Reward: -18.0
Episode: 29, Reward: -20.0
Episode: 30, Reward: -21.0
Episode: 31, Reward: -20.0
Episode: 32, Reward: -21.0
Episode: 33, Reward: -19.0
Episode: 34, Reward: -21.0
Episode: 35, Reward: -18.0
Episode: 36, Reward: -21.0
Episode: 37

INFO:tensorflow:Assets written to: policy_model_100/assets


INFO:tensorflow:Assets written to: policy_model_100/assets


INFO:tensorflow:Assets written to: value_model_afterloadingmodel_100/assets


INFO:tensorflow:Assets written to: value_model_afterloadingmodel_100/assets


...

Episode: 300, Reward: -15.0
Average Reward (Last 100 Episodes): -15.92


INFO:tensorflow:Assets written to: policy_model_300/assets


INFO:tensorflow:Assets written to: policy_model_300/assets


INFO:tensorflow:Assets written to: value_model_afterloadingmodel_300/assets


INFO:tensorflow:Assets written to: value_model_afterloadingmodel_300/assets


In [ ]:
episode_numbers = list(range(1, len(episode_rewards) + 1))
plt.plot(episode_numbers, episode_rewards, label="Episode Reward")
moving_avg = np.convolve(episode_rewards, np.ones(100) / 100, mode='valid')
plt.plot(episode_numbers[-len(moving_avg):], moving_avg, label="Moving Average (100 episodes)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.show()
plt.savefig('episode_rewards_'+ str(batch)+'.png')


## Testing the Model

In [ ]:


env = gym.make("Pong-v0")

episode_totalrewards = []


for episode in range(500):
    #reward_history_eps = []
    terminated = False
    truncated = False
    episode_reward = 0
    state = preprocess(env.reset()[0])

    prev_state = state

    while (not terminated) and (not truncated):
        # sampling an action from the policy
        with open('output.log', 'w') as log_file:
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                if episode_reward==0:
                    combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(prev_state)], axis=-1)
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                else:
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]

        dist = tfp.distributions.Categorical(probs=action_prob, dtype=tf.float32)
        action = dist.sample()
        #print(int(action))
        if int(action)==0:
            a=2
        elif int(action)==1:
            a=3
        next_state, reward, terminated, truncated, info = env.step(a)
        next_state = preprocess(next_state)
        combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(next_state)], axis=-1)
        prev_state = next_state
        #reward_history_eps.append(reward)
        episode_reward = episode_reward+ reward

    episode_totalrewards.append(episode_reward)
    #print(f"Episode: {episode}, Reward: {episode_reward}")


# Close the environment
env.close()


In [ ]:
mean_reward = np.mean(episode_totalrewards)
std_reward = np.std(episode_totalrewards)


# Create a histogram
plt.hist(episode_totalrewards, bins=10, color='orange', alpha=0.7)
plt.xlabel('Episode Total Rewards')
plt.ylabel('Number of Episodes')
plt.title('Histogram of Episode Rewards')
plt.legend([f'Rewards (Mean: {mean_reward:.2f}, Std Dev: {std_reward:.2f})'])
plt.grid(True)




# Show the histogram
plt.show()
